# Structured output

Assistant version 3. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_typed_ticket

Source: [01_typed_ticket.py](examples/01_typed_ticket.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Pydantic primer; structured_output_model on Agent/call and result.structured_output.

Run: uv run python modules/04_structured_output/examples/01_typed_ticket.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.schemas import TicketTriage
from common.testing import tool_message


def main() -> None:
    _, cfg = lesson_settings()
    prediction = {"category": "change", "urgency": "normal", "summary": "Change a standard fare."}
    model = get_model(cfg, responses=[tool_message("TicketTriage", prediction)])
    # KEY IDEA: the model produces a validated object, not JSON we trust without checking.
    agent = Agent(
        model=model,
        structured_output_model=TicketTriage,
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent("Classify: Can I change my standard flight?", limits={"turns": 3})
    console.print(result.structured_output.model_dump(mode="json"))
    complete(result, cfg)


run_lesson(main, "modules/04_structured_output/examples/01_typed_ticket.py")

## 02_nested_itinerary

Source: [02_nested_itinerary.py](examples/02_nested_itinerary.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Nested schemas, enums and optional fields; extract a confirmation email.

Run: uv run python modules/04_structured_output/examples/02_nested_itinerary.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.schemas import Itinerary
from common.testing import tool_message


def main() -> None:
    _, cfg = lesson_settings()
    payload = {
        "traveler": "Demo Traveler",
        "legs": [{"flight_id": "FLT-0001", "destination": "Lisbon", "price_cents": 25000}],
        "total_cents": 25000,
    }
    agent = Agent(
        model=get_model(cfg, responses=[tool_message("Itinerary", payload)]),
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: nested validation checks both field types and relationships between fields.
    result = agent(
        "Extract: Demo Traveler flies FLT-0001 to Lisbon for USD 250. "
        "Flight total USD 250; no hotel booked.",
        structured_output_model=Itinerary,
        limits={"turns": 3},
    )
    console.print(result.structured_output.model_dump(mode="json"))
    complete(result, cfg)


run_lesson(main, "modules/04_structured_output/examples/02_nested_itinerary.py")

## 03_validation_retry

Source: [03_validation_retry.py](examples/03_validation_retry.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Validation failure, correction, retry limits and malformed output tests.

Run: uv run python modules/04_structured_output/examples/03_validation_retry.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.schemas import TicketTriage
from common.testing import tool_message


def main() -> None:
    _, cfg = lesson_settings()
    bad = {"category": "made-up", "urgency": "normal", "summary": "A refund request."}
    fixed = {**bad, "category": "refund"}
    model = get_model(
        cfg,
        responses=[
            tool_message("TicketTriage", bad, "bad"),
            tool_message("TicketTriage", fixed, "fixed"),
        ],
    )
    agent = Agent(
        model=model,
        structured_output_model=TicketTriage,
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: a validation error becomes feedback, but repairs have a turn budget.
    result = agent("Classify: Please refund my booking.", limits={"turns": 3})
    console.print("Validated category:", result.structured_output.category)
    if cfg.provider == "mock":
        assert model.index == 2
        console.print("Invalid category rejected, then corrected.")
    complete(result, cfg)


run_lesson(main, "modules/04_structured_output/examples/03_validation_retry.py")

## 04_structure_with_tools

Source: [04_structure_with_tools.py](examples/04_structure_with_tools.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Structured answers after tool use; labelled ticket accuracy and CSV export.

Run: uv run python modules/04_structured_output/examples/04_structure_with_tools.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.schemas import TicketTriage
from common.testing import tool_message
from common.tools import policy_lookup


def main() -> None:
    _, cfg = lesson_settings()
    model = get_model(
        cfg,
        responses=[
            tool_message("policy_lookup", {}),
            tool_message(
                "TicketTriage",
                {
                    "category": "refund",
                    "urgency": "normal",
                    "summary": "Refund policy needs review.",
                },
            ),
        ],
    )
    # KEY IDEA: ordinary tool results can ground the final structured response.
    agent = Agent(
        model=model,
        tools=[policy_lookup],
        structured_output_model=TicketTriage,
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent(
        "Read the refund policy, then classify: Please refund my booking.", limits={"turns": 4}
    )
    console.print(result.structured_output.model_dump(mode="json"))
    complete(result, cfg)


run_lesson(main, "modules/04_structured_output/examples/04_structure_with_tools.py")

## ticket_triage

Source: [ticket_triage.py](use_case/ticket_triage.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Classify the held-out ticket subset, export CSV, and score after prediction.

Run: uv run python modules/04_structured_output/use_case/ticket_triage.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

import csv

from strands import Agent

from common.config import ROOT
from common.data import load_jsonl
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, estimate_cost, run_cli
from common.schemas import TicketTriage
from common.testing import tool_message
from common.triage import fixture_triage


def main() -> None:
    _, cfg = lesson_settings()
    cases = load_jsonl("eval/ticket_triage.jsonl")
    rows, correct = [], 0
    usage = {"inputTokens": 0, "outputTokens": 0}
    # KEY IDEA: labels are read by the scorer, never put in the agent's prompt.
    for item in cases:
        scripted = fixture_triage(item["input"])
        agent = Agent(
            model=get_model(cfg, responses=[tool_message("TicketTriage", scripted)]),
            structured_output_model=TicketTriage,
            callback_handler=None,
            retry_strategy=None,
        )
        result = agent(
            "Classify this Nimbus ticket. High urgency means departure within 24 hours. "
            + item["input"],
            limits={"turns": 3},
        )
        prediction = result.structured_output
        for key in usage:
            usage[key] += result.metrics.accumulated_usage.get(key, 0)
        if prediction is None:
            raise RuntimeError("No valid classification within the turn budget")
        match = all(getattr(prediction, key) == value for key, value in item["expected"].items())
        correct += int(match)
        rows.append({"id": item["id"], **prediction.model_dump(mode="json"), "correct": match})
    output = ROOT / ".nimbus/ticket_triage.csv"
    output.parent.mkdir(exist_ok=True)
    with output.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    console.print(f"Exact category+urgency accuracy: {correct}/{len(cases)}")
    console.print("Replay validates scoring/CSV only; run live to measure the selected model.")
    console.print("CSV:", str(output))
    console.print("Total token usage:", usage, "Inference estimate USD:", estimate_cost(usage, cfg))
    complete()


run_lesson(main, "modules/04_structured_output/use_case/ticket_triage.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
